# ACE example


In [ ]:
from pathlib import Path
import importlib.util


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "functions").is_dir() and (candidate / "pyspedas").is_dir():
            return candidate
    raise RuntimeError("Could not locate MHDTurbPy root (missing functions/ and pyspedas/).")


root_dir = _find_repo_root(Path.cwd())
path_setup_file = root_dir / "functions" / "path_setup.py"
spec = importlib.util.spec_from_file_location("mhdturbpy_path_setup", path_setup_file)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load path setup from {path_setup_file}")
path_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(path_setup)

root_dir = path_setup.ensure_project_paths(
    start=Path.cwd(),
    include_downloading_helpers=True,
    include_anisotropy_toolbox=True,
    include_sc_pos=True,
)
print(f"Using MHDTurbPy root: {root_dir}")



## Download ACE data


In [ ]:
settings = {
    "Data_path": Path(root_dir).joinpath("data"),
    "save_destination": Path(root_dir).joinpath("examples", "downloaded_intervals"),
    "overwrite_files": 1,
    "save_all": True,
    "addit_time_around": 5,
    "estimate_derived_param": True,
    "apply_hampel": True,
    "hampel_params": {
        "w": 100,
        "std": 3,
    },
    "E_field": {
        "flag": True,
        "new_cadence": "5s",
    },
    "cut_in_small_windows": {
        "flag": False,
        "hours_per_win": 6,
    },
    "sc": "ACE",
    "t0": "2000-04-06 00:00:00",
    "tf": "2000-04-06 06:00:00",
    "credentials": None,
}

Parallel(n_jobs=1)(
    delayed(download.main_function)(
        settings,
        vars_2_downnload=None,
        cdf_lib_path=None,
    )
    for _ in range(1)
)


## Visualize downloaded interval


In [ ]:
plt.close("all")

sc = "ACE"
which_int = 0
load_path = Path(root_dir).joinpath("examples", "downloaded_intervals", sc)
save_path = Path(root_dir).joinpath("selected_intervals")

finnames = func.load_files(load_path, "final.pkl")
gennames = func.load_files(load_path, "general.pkl")
signames = func.load_files(load_path, "sig_c_sig_r.pkl")
maggaps = func.load_files(load_path, "mag_gaps.pkl")
pargaps = func.load_files(load_path, "par_gaps.pkl")

res = interactive_figs.initialize_figures(
    check_exist=False,
    final_Mag_file=finnames[which_int],
    final_Par_file=finnames[which_int],
    general_file=gennames[which_int],
    sig_c_sig_r_file=signames[which_int],
    final_Elec_file=None,
    mag_gaps_file=maggaps[which_int],
    par_gaps_file=pargaps[which_int],
    E_gaps_file=None,
    nn_df_file=None,
    spacecraft=sc,
    only_one_window=True,
    start_time=None,
    end_time=None,
    no_plot=False,
)

(
    fig,
    axes,
    f_names,
    g_names,
    sig_names,
    nn_names,
    n_files,
    ostart,
    oend,
    sf_names,
    diagnostics,
) = res
